In [ ]:
import pandas as pd
import os
from glob import glob

folder = r"/content/drive/MyDrive/tree species countries"
files = glob(os.path.join(folder, "*.csv"))

all_dfs = []

for file in files:
    try:
        try:
            # Try UTF-8 first
            df = pd.read_csv(file, encoding='utf-8')
        except UnicodeDecodeError:
            # If UTF-8 fails, try another common encoding
            df = pd.read_csv(file, encoding='latin1')

        country = os.path.basename(file).replace(".csv", "")
        df["source_country"] = country

        all_dfs.append(df)

        print(f"Loaded: {country}")

    except Exception as e:
        print(f"Error reading {file}: {e}")

merged_df = pd.concat(all_dfs, ignore_index=True)

merged_df.to_csv(
    "/content/drive/MyDrive/merged_native_countries.csv",
    index=False
)

print("Finished.")

Loaded: Jordan
Loaded: Jersey
Loaded: Isle of Man
Loaded: Hungary
Loaded: Ghana
Loaded: Haiti
Loaded: Bangladesh
Loaded: Belize
Loaded: Gambia
Loaded: Honduras
Loaded: Israel
Loaded: Brunei Darussalam
Loaded: Bonaire, Sint Eustatius and Saba
Loaded: Jamaica
Loaded: British Indian Ocean Territory
Loaded: Guyana
Loaded: Belgium
Loaded: Barbados
Loaded: Italy
Loaded: Benin
Loaded: Iceland
Loaded: Bhutan
Loaded: Indonesia
Loaded: Iraq
Loaded: Botswana
Loaded: Bosnia and Herzegovina
Loaded: Brazil
Loaded: Japan
Loaded: Ireland
Loaded: Bolivia (Plurinational State of)
Loaded: Guinea-Bissau
Loaded: Bermuda
Loaded: British Virgin Islands
Loaded: Iran (Islamic Republic of)
Loaded: Guam
Loaded: India
Loaded: Cyprus
Loaded: French Polynesia
Loaded: Estonia
Loaded: Gabon
Loaded: Greece
Loaded: Belarus
Loaded: Argentina
Loaded: Australia
Loaded: Grenada
Loaded: France
Loaded: Bahrain
Loaded: El Salvador
Loaded: Germany
Loaded: Albania
Loaded: Dominica
Loaded: Eswatini
Loaded: Azerbaijan
Loaded: Gua

In [ ]:
import pandas as pd

# Load datasets
flowering = pd.read_csv("/content/drive/MyDrive/flowering_trees.csv")
native = pd.read_csv("/content/drive/MyDrive/merged_native_countries.csv")

# Clean names for matching
flowering["scientificname_clean"] = (
    flowering["scientificname"]
    .astype(str)
    .str.strip()
    .str.lower()
)

native["taxon_clean"] = (
    native["taxon"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Group repeated species and combine countries
native_grouped = (
    native.groupby("taxon_clean")["native to"]
    .apply(
        lambda x: "|".join(
            sorted(
                set(
                    c.strip()
                    for val in x.dropna()
                    for c in str(val).split("|")
                )
            )
        )
    )
    .reset_index()
)

# Merge while keeping ALL flowering tree rows
result = flowering.merge(
    native_grouped,
    left_on="scientificname_clean",
    right_on="taxon_clean",
    how="left"
)

# Optional: mark matched/unmatched
result["native_match"] = result["native to"].notna()

# Remove helper columns
result.drop(
    columns=["scientificname_clean", "taxon_clean"],
    inplace=True
)

# Save final file
result.to_csv(
    "flowering_trees_native_countries.csv",
    index=False
)

print(f"Total flowering species: {len(result)}")
print(f"Matched: {result['native_match'].sum()}")
print(f"Unmatched: {(~result['native_match']).sum()}")
print("Saved successfully")

Total flowering species: 53477
Matched: 53223
Unmatched: 254
Saved successfully


In [ ]:
import pandas as pd

df = pd.read_csv("flowering_trees_native_countries.csv")

countries = set()

for value in df["native to"].dropna():
    for country in str(value).split("|"):
        countries.add(country.strip())

unique_countries = pd.DataFrame(
    sorted(countries),
    columns=["country name"]
)

unique_countries.to_csv("unique_native_countries.csv", index=False)

print(f"{len(unique_countries)} unique countries/territories found")

241 unique countries/territories found


In [ ]:
{
    "Côte dIvoire": "Côte d’Ivoire"
}

{'Côte d\x92Ivoire': 'Côte d’Ivoire'}

In [ ]:
import pandas as pd

# Your country list
ours = pd.read_csv("unique_native_countries.csv")

# UN M49 list
un = pd.read_csv("/content/drive/MyDrive/UNSD — Methodology.csv", sep=';')

ours_set = set(ours["country name"].str.strip())
un_set = set(un["Country or Area"].str.strip())

exact_matches = ours_set & un_set
only_in_ours = ours_set - un_set
only_in_un = un_set - ours_set

print(f"Exact matches: {len(exact_matches)}")
print(f"Need standardization: {len(only_in_ours)}")

for c in sorted(only_in_ours):
    print(c)

Exact matches: 238
Need standardization: 3
Côte dIvoire
Disputed Territory
Taiwan, Province of China


In [ ]:
print(repr("Côte dIvoire"))

'Côte d\x92Ivoire'


Native Range

In [44]:
import pandas as pd

# -----------------------------
# Load flowering trees dataset
# -----------------------------
trees = pd.read_csv(
    "flowering_trees_native_countries.csv"
)

# -----------------------------
# Load UNSD dataset
# -----------------------------
unsd = pd.read_csv(
    "/content/drive/MyDrive/UNSD — Methodology.csv",
    sep=";",
    dtype=str
)

# Remove whitespace from column names
unsd.columns = unsd.columns.str.strip()

# -----------------------------
# Identify required columns
# -----------------------------
country_col = "Country or Area"
subregion_col = "Sub-region Name"
intermediate_col = "Intermediate Region Name"

# Replace NaN with empty strings
unsd[intermediate_col] = unsd[intermediate_col].fillna("").str.strip()
unsd[subregion_col] = unsd[subregion_col].fillna("").str.strip()

# -----------------------------
# Use Intermediate Region if available,
# otherwise use Sub-region
# -----------------------------
unsd["native range"] = unsd.apply(
    lambda row: row[intermediate_col]
    if row[intermediate_col] != ""
    else row[subregion_col],
    axis=1
)

# Build lookup dictionary
country_to_range = dict(
    zip(
        unsd[country_col].str.strip(),
        unsd["native range"]
    )
)

print(f"Loaded {len(country_to_range)} country mappings")

# -----------------------------
# Convert countries to ranges
# -----------------------------
def get_native_range(native_string):

    if pd.isna(native_string):
        return ""

    ranges = []

    for country in str(native_string).split("|"):

        country = country.strip()

        if country in country_to_range:
            ranges.append(country_to_range[country])

    # Remove duplicates while preserving order
    ranges = list(dict.fromkeys(ranges))

    return "|".join(ranges)

# Change "native_to" if your column has a different name
trees["native range"] = trees["native to"].apply(
    get_native_range
)

# -----------------------------
# Find unmatched countries
# -----------------------------
all_countries = set()

for value in trees["native to"].dropna():
    for country in str(value).split("|"):
        all_countries.add(country.strip())

unmatched = sorted(
    c for c in all_countries
    if c not in country_to_range
)

print("\nUnmatched countries:")
for country in unmatched:
    print(country)

print(f"\nTotal unmatched: {len(unmatched)}")

# -----------------------------
# Save output
# -----------------------------
trees.to_csv(
    "flowering_trees_native_range.csv",
    index=False
)

print("\nSaved: flowering_trees_native_range.csv")

Loaded 248 country mappings

Unmatched countries:
Côte dIvoire
Disputed Territory
Taiwan, Province of China

Total unmatched: 3

Saved: flowering_trees_native_range.csv
